# 1000-year balanced repeat-year control (RYF 1990-91)

Every trained emulator of an intercomparison run (default `intercomparison_90ep_v3`) is driven by ACCESS-OM2's repeat-year forcing year, **May 1990 - April 1991**, repeated for 1000 years from the true April 1990 state.

**Balanced:** the year's net global heat and freshwater fluxes are removed (a uniform offset over the ocean, length-weighted so the year integrates to zero), in both the network's input and the budget constraint. A constrained emulator's global content therefore returns to the same value every year; anything that drifts, grows or collapses is the emulator's own behaviour. The unconstrained Samudra-style baseline gets the same balanced input.

Each emulator is rebuilt from its training stage's `config.json` and `weights.pt`, so it is exactly the model that was trained and evaluated. Results are saved per emulator to `runs/control_ryf_<years>yr/<name>.nc`; an emulator whose file exists is not rerun.

In [ ]:
# =============================================================================
# 1. Setup
# =============================================================================
import os
import sys
from pathlib import Path

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import matplotlib.pyplot as plt
import torch
import xarray as xr

REPO_ROOT = Path("/home/156/txs156/uom/OM2-emulator")
sys.path.insert(0, str(REPO_ROOT / "src"))
from Experiment import (
    ExperimentConfig, build_data, build_model, build_diffusion,
    run_repeat_year_control, reference_rms, plot_long_controls, plot_long_control_spectra,
)

TRAINED = REPO_ROOT / "runs" / os.environ.get("CONTROL_SOURCE_RUN", "intercomparison_90ep_v3")
YEARS = int(os.environ.get("CONTROL_YEARS", "1000"))
FIRST_MONTH = "1990-05"   # ACCESS-OM2 RYF year: May 1990 - April 1991
OUT = REPO_ROOT / "runs" / f"control_ryf_{YEARS}yr"
FIG_DIR = REPO_ROOT / "figures" / f"control_ryf_{YEARS}yr"
OUT.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Emulator -> (training stage folder, members). Main text: samudra (ConvNeXt + constraint),
# deterministic, crps_ensemble; the rest are supplementary.
EMULATORS = {
    "samudra": ("samudra", 1),
    "deterministic": ("deterministic", 1),
    "crps_ensemble": ("crps_ensemble", 10),
    "samudra_crps": ("samudra_crps", 10),
    "samudra_style": ("samudra_style", 1),
    "diffusion_onestep": ("diffusion", 3),
    "diffusion": ("diffusion_pushforward", 3),
}
MAIN_TEXT = ["samudra", "deterministic", "crps_ensemble"]


def save(fig, name):
    fig.savefig(FIG_DIR / name, dpi=150, bbox_inches="tight")
    print(f"Saved {FIG_DIR / name}")


print(f"trained: {TRAINED} | years: {YEARS} | out: {OUT} | device: {DEVICE}")

In [ ]:
# =============================================================================
# 2. Data (the trained run's own data settings, so the cached fields are reused)
# =============================================================================
det_cfg = ExperimentConfig.load(TRAINED / "deterministic" / "config.json")
data = build_data(det_cfg)


def load_emulator(stage):
    """The emulator of a training stage, rebuilt from its config.json and weights.pt."""
    cfg = ExperimentConfig.load(TRAINED / stage / "config.json")
    weights = torch.load(TRAINED / stage / "weights.pt", map_location="cpu")
    if cfg.paradigm == "diffusion":
        mean_model = build_model(det_cfg, data)
        mean_model.load_state_dict(torch.load(TRAINED / "deterministic" / "weights.pt", map_location="cpu"))
        model = build_diffusion(cfg, data, mean_model.to(DEVICE), sigma_data_batches=1)
    else:
        model = build_model(cfg, data)
    model.load_state_dict(weights)
    return cfg, model.to(DEVICE)

In [ ]:
# =============================================================================
# 3. The runs (one file per emulator; existing files are kept)
# =============================================================================
controls = {}
for name, (stage, members) in EMULATORS.items():
    path = OUT / f"{name}.nc"
    if path.exists():
        controls[name] = xr.open_dataset(path).load()
        print(f"{name}: loaded {path}")
        continue
    cfg, model = load_emulator(stage)
    ds = run_repeat_year_control(cfg, model, data, first_month=FIRST_MONTH, years=YEARS, members=members,
                                 snapshot_years=(1, 10, 100, YEARS))
    ds.to_netcdf(path)
    controls[name] = ds
    print(f"{name}: {members} member(s), {YEARS} years, offsets {ds.attrs['flux_offsets']} -> {path}")
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
# =============================================================================
# 4. Figures: global content, RMS anomaly, max |z|, decadal pattern drift; spectra
# =============================================================================
variables = list(next(iter(controls.values())).variable.values)
reference = {v: reference_rms(data, v) for v in variables}
main = {name: controls[name] for name in MAIN_TEXT}
save(plot_long_controls(main, reference), "control_main.png")
save(plot_long_control_spectra(main), "control_spectra_main.png")
save(plot_long_controls(controls, reference), "control_all.png")
save(plot_long_control_spectra(controls), "control_spectra_all.png")

In [ ]:
# =============================================================================
# 5. Summary: first and last decade of each emulator
# =============================================================================
import numpy as np
import pandas as pd

rows = []
for name, ds in controls.items():
    for variable in variables:
        d = ds.sel(variable=variable)
        first, last = slice(0, 120), slice(-120, None)
        dec = d["decadal_mean"]
        w = ds["area"].where(dec.isel(decade=0).notnull(), 0.0)
        ocean_rms = lambda x: float(np.sqrt((x.fillna(0.0) ** 2 * w).sum() / w.sum()))
        rows.append({
            "emulator": name, "variable": variable, "members": d.member.size,
            "rms_first_decade / reference": float(d["rms"].isel(step=first).mean()) / reference[variable],
            "rms_last_decade / reference": float(d["rms"].isel(step=last).mean()) / reference[variable],
            "max_abs_z_last_decade": float(d["max_abs_z"].isel(step=last).max()),
            "global_drift_first_to_last_decade": float(d["global"].isel(step=last).mean() - d["global"].isel(step=first).mean()),
            "pattern_drift_last_vs_first_decade": ocean_rms(dec.isel(decade=-1) - dec.isel(decade=0)) / ocean_rms(dec.isel(decade=0)),
        })
summary = pd.DataFrame(rows).set_index(["variable", "emulator"]).sort_index()
display(summary.round(3))
summary.to_csv(FIG_DIR / "control_summary.csv")